In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

c:\Users\khana\OneDrive\Desktop\GPU t est in Tensorflow\.venv\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.10). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [4]:
# Load CSV file
df = pd.read_csv(r"D:\LLM\LLM_prompt\Dataset\train_essays_7_prompts.csv")

# Combine question and answer
texts = (df.iloc[:,0].astype(str) + " <SEP> " + df.iloc[:,1].astype(str)).tolist()

print("Number of samples:", len(texts))
print(df.columns)

Number of samples: 14877
Index(['text', 'label'], dtype='object')


In [5]:
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)

Vocabulary size: 61291


In [6]:
# STEP 4: CREATE TRAINING SEQUENCES (FIXED)
input_sequences = []

for line in texts:
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

# LIMIT MAX LENGTH (IMPORTANT FIX)
max_len = 100

input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_len,
    padding='pre',
    truncating='pre'
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

In [7]:
model = Sequential([
    Embedding(input_dim=total_words,
              output_dim=128,
              input_length=max_len-1),

    LSTM(256),

    Dense(total_words, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 99, 128)           7845248   
                                                                 
 lstm (LSTM)                 (None, 256)               394240    
                                                                 
 dense (Dense)               (None, 61291)             15751787  
                                                                 
Total params: 23,991,275
Trainable params: 23,991,275
Non-trainable params: 0
_________________________________________________________________


In [8]:
history = model.fit(
    X,
    y,
    epochs=2,
    batch_size=32,
    verbose=1
)

Epoch 1/2
196183/196183 [==============================] - 9640s 49ms/step - loss: 4.2347 - accuracy: 0.2931
Epoch 2/2
196183/196183 [==============================] - 9508s 48ms/step - loss: 3.8721 - accuracy: 0.3347


In [9]:
model.save("chatbot_model.keras")

In [10]:
def chatbot(prompt, max_words=30):

    text = prompt

    for _ in range(max_words):

        seq = tokenizer.texts_to_sequences([text])[0]
        seq = pad_sequences([seq], maxlen=max_len-1, padding='pre')

        pred = np.argmax(model.predict(seq, verbose=0), axis=-1)[0]

        output_word = ""

        for word, index in tokenizer.word_index.items():
            if index == pred:
                output_word = word
                break

        text += " " + output_word

        if output_word == "<SEP>":
            break

    response = text.split("<SEP>")[-1]
    return response.strip()

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    reply = chatbot(user_input)

    print("Bot:", reply)

Bot: give me a prompt and i think that the face on mars is a natural landform i believe that the face is just a natural landform because it is a natural landform that is
Bot: give me a prompt for code that the author of the challenge of exploring venus the author supports his idea well by explaining why venus is a worthy pursuit despite the dangers it presents the author
Bot: the author supports his idea well by explaining why venus is a worthy pursuit despite the dangers it presents the author supports the idea of studying venus is a worthy
Bot: the author supports his idea well by explaining why venus is a worthy pursuit despite the dangers it presents the author supports the idea of studying venus is a worthy
